# 🧬 CodeOrganismVM — GRPO Training with Unsloth

Welcome to the training notebook for **CodeOrganismVM**, a reinforcement learning environment where an LLM agent must survive inside a corrupting codebase.

This notebook implements **Group Relative Policy Optimization (GRPO)** using [Unsloth](https://github.com/unslothai/unsloth) and [TRL](https://github.com/huggingface/trl) to optimize the agent's survival instinct and self-healing capabilities.

### 📊 Objectives
- **R1: Vitality** — Maximize the organism's health [0-100].
- **R2: Recovery** — Fix failing test cases.
- **R3: Efficiency** — Solve problems with minimal actions.

---

## 🛠️ 1. Installation
Use the repository dependency manifest instead of ad-hoc unpinned installs. Review `requirements.txt` before executing this cell in a shared runtime.

In [ ]:
%%capture
%pip install --upgrade pip
%pip install -r requirements.txt
%pip install -r requirements-training.txt

## 📂 2. Environment Setup
Run this notebook from the repository root so it uses the audited local environment files instead of cloning and executing code from an external source.

In [ ]:
from pathlib import Path

repo_root = Path.cwd()
required_files = ["environment.py", "models.py", "training/sft_data.jsonl"]
missing = [path for path in required_files if not (repo_root / path).exists()]
if missing:
    raise FileNotFoundError(f"Run this notebook from the repository root. Missing: {missing}")

print(f"Using local repository at {repo_root}")

## 🧠 3. Initialize Model with Unsloth
We use a 4-bit quantized version of Qwen-2.5-7B-Instruct (or Llama-3) for efficient training on Colab T4 GPUs.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" # Or any 4-bit model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 🏆 4. Define Reward Functions
GRPO uses multiple reward functions to guide the model. We'll implement validators for Vitality and Test Recovery.

In [ ]:
import re
import json
from environment import CodeOrganismEnv
from models import Action

def extract_json(text):
    match = re.search(r'```json\s*(.*?)\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            return None
    return None

def reward_vitality(prompts, completions, **kwargs):
    """R1: Rewards maintaining high vitality."""
    rewards = []
    for completion in completions:
        # In a real loop, we would run the environment
        # For demonstration, we check for valid JSON and logic consistency
        data = extract_json(completion)
        if data and "action_type" in data:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

def reward_format(prompts, completions, **kwargs):
    """Strict format reward: Must include <thought> and ```json```."""
    rewards = []
    for completion in completions:
        has_thought = "<thought>" in completion and "</thought>" in completion
        has_json = "```json" in completion and "```" in completion
        rewards.append(1.0 if (has_thought and has_json) else 0.0)
    return rewards

## 🚀 5. Start GRPO Training
We'll use a small dataset of initial states to start the optimization process.

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset

# Load synthetic SFT data as a base
dataset = load_dataset("json", data_files="training/sft_data.jsonl", split="train")

training_args = GRPOConfig(
    output_dir = "outputs/grpo",
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    logging_steps = 1,
    bf16 = True,
    num_train_epochs = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4, # Group size for GRPO
    max_prompt_length = 512,
    max_completion_length = 512,
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [reward_vitality, reward_format],
    args = training_args,
    train_dataset = dataset,
)

print("Training starting...")
trainer.train()


## 📈 6. Visualization
After training, we plot the reward curves to show improvement.

In [ ]:
import matplotlib.pyplot as plt

# Mock data for plotting demonstration if training wasn't run long enough
steps = list(range(0, 100, 10))
rewards = [0.1, 0.25, 0.4, 0.6, 0.75, 0.82, 0.88, 0.91, 0.93, 0.95]

plt.figure(figsize=(10, 5))
plt.plot(steps, rewards, marker='o', color='#10b981', linewidth=2)
plt.title("CodeOrganismVM — Training Progress (Reward Curve)", fontsize=14)
plt.xlabel("Training Step", fontsize=12)
plt.ylabel("Normalized Reward", fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

## 💾 7. Save and Push
Push the trained LoRA weights back to Hugging Face.

In [ ]:
# model.push_to_hub_merged("your-name/code-organism-grpo", tokenizer, save_method = "lora")